# Imports

In [ ]:
# importing different packages
import pandas as pd
import pickle
import geopandas as gpd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import numpy as np
from shapely.geometry import Point


In [ ]:
dataframes = []
capacity_names = ['nameplate_capacity_mw',
                  'project_capacity_mw',
                  'total_capacity_mw',
                  'total_capacity_mw']

In [ ]:
for df in dataframes:
    print(df["nameplate_capacity_mw"].describe())
    print(df.groupby('zip_code')['nameplate_capacity_mw'].sum().describe())

    zip_counts = df['zip_code'].value_counts()
    top_10_zipcodes = zip_counts.head(20)
    plt.figure(figsize=(10, 6))
    top_10_zipcodes.plot(kind='bar')
    plt.title('Top 20 Most Frequent ZIP Codes')
    plt.xlabel('ZIP Code')
    plt.ylabel('Frequency')
    plt.xticks(rotation=45)
    plt.show()

    #
    aggregated = df.groupby('zip_code')['nameplate_capacity_mw'].sum().sort_values(ascending=False).head(20)
    aggregated.plot(kind='bar')
    plt.title('Top 20 Highest Capacity ZIP Codes')
    plt.xlabel('ZIP Code')
    plt.ylabel('Capacity (MW)')
    plt.xticks(rotation=45)
    plt.show()

    #
    aggregated = df.groupby('zip_code')['nameplate_capacity_mw'].sum().reset_index()
    zip_shapefile = '/Users/danielleknutson/Downloads/tl_2024_us_zcta520/tl_2024_us_zcta520.shp'
    gdf_zipcodes = gpd.read_file(zip_shapefile)
    states = gpd.read_file("https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json")
    california = states[states['name'] == 'California']
    gdf_zipcodes = gdf_zipcodes[gdf_zipcodes.geometry.within(california.geometry.iloc[0])]
    gdf_zipcodes['ZCTA5CE20'] = gdf_zipcodes['ZCTA5CE20'].astype(str).str.zfill(5)
    aggregated['zip_code'] = aggregated['zip_code'].astype(str).str.zfill(5)
    gdf_zipcodes = gdf_zipcodes.merge(aggregated, left_on='ZCTA5CE20', right_on='zip_code', how='left')
    gdf_zipcodes['nameplate_capacity_mw'] = gdf_zipcodes['nameplate_capacity_mw'].fillna(0)
    fig, ax = plt.subplots(figsize=(12, 12))
    gdf_zipcodes.plot(ax=ax, column='nameplate_capacity_mw', cmap='viridis', legend=True,
                    legend_kwds={'label': "Capacity by ZIP Code",
                                'orientation': "horizontal"},
                    edgecolor='gray', linewidth=0.5, alpha=0.8)
    gdf_zipcodes[gdf_zipcodes['nameplate_capacity_mw'] == 0].plot(ax=ax, color='lightgray', edgecolor='gray', linewidth=0.2)
    plt.title('California ZIP Code Capacity Distribution', fontsize=16)
    plt.xlabel('Longitude', fontsize=12)
    plt.ylabel('Latitude', fontsize=12)
    plt.show()

    #
    zip_code_counts = df['zip_code'].value_counts().reset_index()
    zip_code_counts.columns = ['zip_code', 'occurrences']
    zip_shapefile = '/Users/danielleknutson/Downloads/tl_2024_us_zcta520/tl_2024_us_zcta520.shp'
    gdf_zipcodes = gpd.read_file(zip_shapefile)
    gdf_zipcodes['ZCTA5CE20'] = gdf_zipcodes['ZCTA5CE20'].astype(str)  # Ensure matching formats
    zip_code_counts['zip_code'] = zip_code_counts['zip_code'].astype(str)  # Ensure matching formats
    gdf_zipcodes = gdf_zipcodes.merge(zip_code_counts, left_on='ZCTA5CE20', right_on='zip_code', how='left')
    fig, ax = plt.subplots(figsize=(12, 12))
    gdf_zipcodes.plot(ax=ax, column='occurrences', cmap='viridis', legend=True,
                    legend_kwds={'label': "Number of Occurrences by ZIP Code",
                                'orientation': "horizontal"},
                    edgecolor='gray', linewidth=0.5, alpha=0.8)
    plt.title('Number of Occurrences of Each ZIP Code in California', fontsize=16)
    plt.xlabel('Longitude', fontsize=12)
    plt.ylabel('Latitude', fontsize=12)
    plt.show()

    #
    acs_data['zip_code'] = acs_data['zip_code'].astype(str).str.zfill(5)  # Ensure format is matching
    aggregated['zip_code'] = aggregated['zip_code'].astype(str).str.zfill(5)  # Ensure format is matching
    merged_data = acs_data.merge(aggregated, on='zip_code', how='left')
    merged_data['total_population'] = merged_data['total_population'].replace(0, 1).fillna(1)
    merged_data['der_capacity_per_capita'] = merged_data['nameplate_capacity_mw'] / merged_data['total_population']
    zip_shapefile = '/Users/danielleknutson/Downloads/tl_2024_us_zcta520/tl_2024_us_zcta520.shp'
    gdf_zipcodes = gpd.read_file(zip_shapefile)
    states = gpd.read_file("https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json")
    california = states[states['name'] == 'California']
    gdf_zipcodes = gdf_zipcodes[gdf_zipcodes.geometry.within(california.geometry.iloc[0])]
    gdf_zipcodes['ZCTA5CE20'] = gdf_zipcodes['ZCTA5CE20'].astype(str).str.zfill(5)
    merged_data['zip_code'] = merged_data['zip_code'].astype(str).str.zfill(5)
    gdf_zipcodes = gdf_zipcodes.merge(merged_data, left_on='ZCTA5CE20', right_on='zip_code', how='left')
    gdf_zipcodes['der_capacity_per_capita'] = gdf_zipcodes['der_capacity_per_capita'].fillna(0)
    fig, ax = plt.subplots(figsize=(12, 12))
    gdf_zipcodes.plot(ax=ax, column='der_capacity_per_capita', cmap='cool', legend=True,
                    legend_kwds={'label': "DER Capacity per Capita by ZIP Code",
                                'orientation': "horizontal"},
                    edgecolor='gray', linewidth=0.5, alpha=0.8)
    gdf_zipcodes[gdf_zipcodes['der_capacity_per_capita'] == 0].plot(ax=ax, color='lightgray', edgecolor='gray', linewidth=0.2)

    plt.title('California ZIP Code Storage DER Capacity per Capita Distribution', fontsize=16)
    plt.xlabel('Longitude', fontsize=12)
    plt.ylabel('Latitude', fontsize=12)
    plt.show()


    #
    plotbigpercapita = merged_data.groupby('zip_code')['der_capacity_per_capita'].sum().sort_values(ascending=False).head(20)
    plotbigpercapita.plot(kind='bar')
    plt.title('Top 20 Highest Capacity census_tract')
    plt.xlabel('ZIP Code')
    plt.ylabel('Capacity (MW)')
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
# print(uswtdb_wind_der["project_capacity_mw"].describe())
# print(uswtdb_wind_der.groupby('zip_code')['project_capacity_mw'].sum().describe())

# zip_counts = uswtdb_wind_der['zip_code'].value_counts()
# top_10_zipcodes = zip_counts.head(20)
# plt.figure(figsize=(10, 6))
# top_10_zipcodes.plot(kind='bar')
# plt.title('Top 20 Most Frequent ZIP Codes')
# plt.xlabel('ZIP Code')
# plt.ylabel('Frequency')
# plt.xticks(rotation=45)
# plt.show()


# aggregated_uswtdb_wind_der = uswtdb_wind_der.groupby('zip_code')['project_capacity_mw'].sum().sort_values(ascending=False).head(20)
# aggregated_uswtdb_wind_der.plot(kind='bar')
# plt.title('Top 20 Highest Capacity ZIP Codes')
# plt.xlabel('ZIP Code')
# plt.ylabel('Capacity (MW)')
# plt.xticks(rotation=45)
# plt.show()


# aggregated_uswtdb_wind_der = uswtdb_wind_der.groupby('zip_code')['project_capacity_mw'].sum().reset_index()
# zip_shapefile = '/Users/danielleknutson/Downloads/tl_2024_us_zcta520/tl_2024_us_zcta520.shp'
# gdf_zipcodes = gpd.read_file(zip_shapefile)
# states = gpd.read_file("https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json")
# california = states[states['name'] == 'California']
# gdf_zipcodes = gdf_zipcodes[gdf_zipcodes.geometry.within(california.geometry.iloc[0])]
# gdf_zipcodes['ZCTA5CE20'] = gdf_zipcodes['ZCTA5CE20'].astype(str).str.zfill(5)
# aggregated_uswtdb_wind_der['zip_code'] = aggregated_uswtdb_wind_der['zip_code'].astype(str).str.zfill(5)
# gdf_zipcodes = gdf_zipcodes.merge(aggregated_uswtdb_wind_der, left_on='ZCTA5CE20', right_on='zip_code', how='left')
# gdf_zipcodes['project_capacity_mw'] = gdf_zipcodes['project_capacity_mw'].fillna(0)
# fig, ax = plt.subplots(figsize=(12, 12))
# gdf_zipcodes.plot(ax=ax, column='project_capacity_mw', cmap='viridis', legend=True,
#                   legend_kwds={'label': "Capacity by ZIP Code",
#                                'orientation': "horizontal"},
#                   edgecolor='gray', linewidth=0.5, alpha=0.8)
# gdf_zipcodes[gdf_zipcodes['project_capacity_mw'] == 0].plot(ax=ax, color='lightgray', edgecolor='gray', linewidth=0.2)
# plt.title('California ZIP Code Capacity Distribution', fontsize=16)
# plt.xlabel('Longitude', fontsize=12)
# plt.ylabel('Latitude', fontsize=12)
# plt.show()



# zip_shapefile = '/Users/danielleknutson/Downloads/tl_2024_us_zcta520/tl_2024_us_zcta520.shp'
# gdf_zipcodes = gpd.read_file(zip_shapefile)
# states = gpd.read_file("https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json")
# california = states[states['name'] == 'California']
# gdf_zipcodes = gdf_zipcodes[gdf_zipcodes.geometry.within(california.geometry.iloc[0])]
# zip_code_counts = zip_code_counts.rename(columns={'zip_code': 'ZCTA5CE20'})  # Ensure matching columns
# gdf_zipcodes['ZCTA5CE20'] = gdf_zipcodes['ZCTA5CE20'].astype(str)  # Ensure the formats match
# zip_code_counts['ZCTA5CE20'] = zip_code_counts['ZCTA5CE20'].astype(str)  # Ensure the formats match
# gdf_zipcodes = gdf_zipcodes.merge(zip_code_counts, on='ZCTA5CE20', how='left')
# gdf_zipcodes['occurrences'] = gdf_zipcodes['occurrences'].fillna(0)
# fig, ax = plt.subplots(figsize=(12, 12))
# gdf_zipcodes.plot(ax=ax, column='occurrences', cmap='viridis', legend=True,
#                   legend_kwds={'label': "Number of Occurrences by ZIP Code", 'orientation': "horizontal"},
#                   edgecolor='black', linewidth=0.8, alpha=0.8)  # Thicker black outlines for ZIP codes
# gdf_zipcodes[gdf_zipcodes['occurrences'] == 0].plot(ax=ax, color='lightgray', edgecolor='gray', linewidth=0.2)
# plt.title('Number of Occurrences of Each ZIP Code in California', fontsize=16)
# plt.xlabel('Longitude', fontsize=12)
# plt.ylabel('Latitude', fontsize=12)
# plt.show()

In [ ]:
# acs_data['zip_code'] = acs_data['zip_code'].astype(str).str.zfill(5)  # Ensure format is matching
# aggregated_uswtdb_wind_der['zip_code'] = aggregated_uswtdb_wind_der['zip_code'].astype(str).str.zfill(5)  # Ensure format is matching
# merged_data = acs_data.merge(aggregated_uswtdb_wind_der, on='zip_code', how='left')
# merged_data['total_population'] = merged_data['total_population'].replace(0, 1).fillna(1)
# merged_data['der_capacity_per_capita'] = merged_data['project_capacity_mw'] / merged_data['total_population']
# zip_shapefile = '/Users/danielleknutson/Downloads/tl_2024_us_zcta520/tl_2024_us_zcta520.shp'
# gdf_zipcodes = gpd.read_file(zip_shapefile)
# states = gpd.read_file("https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json")
# california = states[states['name'] == 'California']
# gdf_zipcodes = gdf_zipcodes[gdf_zipcodes.geometry.within(california.geometry.iloc[0])]
# gdf_zipcodes['ZCTA5CE20'] = gdf_zipcodes['ZCTA5CE20'].astype(str).str.zfill(5)
# merged_data['zip_code'] = merged_data['zip_code'].astype(str).str.zfill(5)
# gdf_zipcodes = gdf_zipcodes.merge(merged_data, left_on='ZCTA5CE20', right_on='zip_code', how='left')
# gdf_zipcodes['der_capacity_per_capita'] = gdf_zipcodes['der_capacity_per_capita'].fillna(0)
# fig, ax = plt.subplots(figsize=(12, 12))
# gdf_zipcodes.plot(ax=ax, column='der_capacity_per_capita', cmap='viridis', legend=True,
#                   legend_kwds={'label': "DER Capacity per Capita by ZIP Code",
#                                'orientation': "horizontal"},
#                   edgecolor='gray', linewidth=0.5, alpha=0.8)
# gdf_zipcodes[gdf_zipcodes['der_capacity_per_capita'] == 0].plot(ax=ax, color='lightgray', edgecolor='gray', linewidth=0.2)

# plt.title('California ZIP Code Wind Turbine DER Capacity per Capita Distribution', fontsize=16)
# plt.xlabel('Longitude', fontsize=12)
# plt.ylabel('Latitude', fontsize=12)
# plt.show()

In [ ]:
# print(power_plant_utility["total_capacity_mw"].describe())
# print(power_plant_utility.groupby('zip_code')['total_capacity_mw'].sum().describe())

# zip_counts = power_plant_utility['zip_code'].value_counts()
# top_10_zipcodes = zip_counts.head(20)
# plt.figure(figsize=(10, 6))
# top_10_zipcodes.plot(kind='bar')
# plt.title('Top 20 Most Frequent ZIP Codes')
# plt.xlabel('ZIP Code')
# plt.ylabel('Frequency')
# plt.xticks(rotation=45)
# plt.show()


# aggregated_power_plant_utility = power_plant_utility.groupby('zip_code')['total_capacity_mw'].sum().sort_values(ascending=False).head(20)
# aggregated_power_plant_utility.plot(kind='bar')
# plt.title('Top 20 Highest Capacity ZIP Codes')
# plt.xlabel('ZIP Code')
# plt.ylabel('Capacity (MW)')
# plt.xticks(rotation=45)
# plt.show()


# aggregated_power_plant_utility = power_plant_utility.groupby('zip_code')['total_capacity_mw'].sum().reset_index()
# zip_shapefile = '/Users/danielleknutson/Downloads/tl_2024_us_zcta520/tl_2024_us_zcta520.shp'
# gdf_zipcodes = gpd.read_file(zip_shapefile)
# states = gpd.read_file("https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json")
# california = states[states['name'] == 'California']
# gdf_zipcodes = gdf_zipcodes[gdf_zipcodes.geometry.within(california.geometry.iloc[0])]
# gdf_zipcodes['ZCTA5CE20'] = gdf_zipcodes['ZCTA5CE20'].astype(str).str.zfill(5)
# aggregated_power_plant_utility['zip_code'] = aggregated_power_plant_utility['zip_code'].astype(str).str.zfill(5)
# gdf_zipcodes = gdf_zipcodes.merge(aggregated_power_plant_utility, left_on='ZCTA5CE20', right_on='zip_code', how='left')
# gdf_zipcodes['total_capacity_mw'] = gdf_zipcodes['total_capacity_mw'].fillna(0)
# fig, ax = plt.subplots(figsize=(12, 12))
# gdf_zipcodes.plot(ax=ax, column='total_capacity_mw', cmap='viridis', legend=True,
#                   legend_kwds={'label': "Capacity by ZIP Code",
#                                'orientation': "horizontal"},
#                   edgecolor='gray', linewidth=0.5, alpha=0.8)
# gdf_zipcodes[gdf_zipcodes['total_capacity_mw'] == 0].plot(ax=ax, color='lightgray', edgecolor='gray', linewidth=0.2)
# plt.title('California ZIP Code Capacity Distribution', fontsize=16)
# plt.xlabel('Longitude', fontsize=12)
# plt.ylabel('Latitude', fontsize=12)
# plt.show()



# zip_shapefile = '/Users/danielleknutson/Downloads/tl_2024_us_zcta520/tl_2024_us_zcta520.shp'
# gdf_zipcodes = gpd.read_file(zip_shapefile)
# states = gpd.read_file("https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json")
# california = states[states['name'] == 'California']
# gdf_zipcodes = gdf_zipcodes[gdf_zipcodes.geometry.within(california.geometry.iloc[0])]
# zip_code_counts = zip_code_counts.rename(columns={'zip_code': 'ZCTA5CE20'})  # Ensure matching columns
# gdf_zipcodes['ZCTA5CE20'] = gdf_zipcodes['ZCTA5CE20'].astype(str)  # Ensure the formats match
# zip_code_counts['ZCTA5CE20'] = zip_code_counts['ZCTA5CE20'].astype(str)  # Ensure the formats match
# gdf_zipcodes = gdf_zipcodes.merge(zip_code_counts, on='ZCTA5CE20', how='left')
# gdf_zipcodes['occurrences'] = gdf_zipcodes['occurrences'].fillna(0)
# fig, ax = plt.subplots(figsize=(12, 12))
# gdf_zipcodes.plot(ax=ax, column='occurrences', cmap='viridis', legend=True,
#                   legend_kwds={'label': "Number of Occurrences by ZIP Code", 'orientation': "horizontal"},
#                   edgecolor='black', linewidth=0.8, alpha=0.8)  # Thicker black outlines for ZIP codes
# gdf_zipcodes[gdf_zipcodes['occurrences'] == 0].plot(ax=ax, color='lightgray', edgecolor='gray', linewidth=0.2)
# plt.title('Number of Occurrences of Each ZIP Code in California', fontsize=16)
# plt.xlabel('Longitude', fontsize=12)
# plt.ylabel('Latitude', fontsize=12)
# plt.show()

In [ ]:
# acs_data['zip_code'] = acs_data['zip_code'].astype(str).str.zfill(5)  # Ensure format is matching
# aggregated_power_plant_utility['zip_code'] = aggregated_power_plant_utility['zip_code'].astype(str).str.zfill(5)  # Ensure format is matching
# merged_data = acs_data.merge(aggregated_power_plant_utility, on='zip_code', how='left')
# merged_data['total_population'] = merged_data['total_population'].replace(0, 1).fillna(1)
# merged_data['der_capacity_per_capita'] = merged_data['total_capacity_mw'] / merged_data['total_population']
# zip_shapefile = '/Users/danielleknutson/Downloads/tl_2024_us_zcta520/tl_2024_us_zcta520.shp'
# gdf_zipcodes = gpd.read_file(zip_shapefile)
# states = gpd.read_file("https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json")
# california = states[states['name'] == 'California']
# gdf_zipcodes = gdf_zipcodes[gdf_zipcodes.geometry.within(california.geometry.iloc[0])]
# gdf_zipcodes['ZCTA5CE20'] = gdf_zipcodes['ZCTA5CE20'].astype(str).str.zfill(5)
# merged_data['zip_code'] = merged_data['zip_code'].astype(str).str.zfill(5)
# gdf_zipcodes = gdf_zipcodes.merge(merged_data, left_on='ZCTA5CE20', right_on='zip_code', how='left')
# gdf_zipcodes['der_capacity_per_capita'] = gdf_zipcodes['der_capacity_per_capita'].fillna(0)
# fig, ax = plt.subplots(figsize=(12, 12))
# gdf_zipcodes.plot(ax=ax, column='der_capacity_per_capita', cmap='viridis', legend=True,
#                   legend_kwds={'label': "DER Capacity per Capita by ZIP Code",
#                                'orientation': "horizontal"},
#                   edgecolor='gray', linewidth=0.5, alpha=0.8)
# gdf_zipcodes[gdf_zipcodes['der_capacity_per_capita'] == 0].plot(ax=ax, color='lightgray', edgecolor='gray', linewidth=0.2)

# plt.title('California ZIP Code Power Plant Capacity per Capita Distribution', fontsize=16)
# plt.xlabel('Longitude', fontsize=12)
# plt.ylabel('Latitude', fontsize=12)
# plt.show()

In [ ]:
# zip_counts = ev_chargers['zip_code'].value_counts()
# top_10_zipcodes = zip_counts.head(20)
# plt.figure(figsize=(10, 6))
# top_10_zipcodes.plot(kind='bar')
# plt.title('Top 20 Most Frequent ZIP Codes')
# plt.xlabel('ZIP Code')
# plt.ylabel('Frequency')
# plt.xticks(rotation=45)
# plt.show()


# # aggregated_ev_chargers= ev_chargers.groupby('zip_code')['total_capacity_mw'].sum().sort_values(ascending=False).head(20)
# # aggregated_ev_chargers.plot(kind='bar')
# # plt.title('Top 20 Highest Capacity ZIP Codes')
# # plt.xlabel('ZIP Code')
# # plt.ylabel('Capacity (MW)')
# # plt.xticks(rotation=45)
# # plt.show()


# zip_shapefile = '/Users/danielleknutson/Downloads/tl_2024_us_zcta520/tl_2024_us_zcta520.shp'
# gdf_zipcodes = gpd.read_file(zip_shapefile)
# states = gpd.read_file("https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json")
# california = states[states['name'] == 'California']
# gdf_zipcodes = gdf_zipcodes[gdf_zipcodes.geometry.within(california.geometry.iloc[0])]
# gdf_zipcodes['ZCTA5CE20'] = gdf_zipcodes['ZCTA5CE20'].astype(str)  # Ensure the formats match
# aggregated_ev["zip_code"] = aggregated_ev["zip_code"].astype(str)
# aggregated_ev.rename(columns={"zip_code": "ZCTA5CE20"}, inplace= True)
# gdf_zipcodes = gdf_zipcodes.merge(aggregated_ev, on='ZCTA5CE20', how='left')
# for i in type_stats:
#     gdf_zipcodes[i] = gdf_zipcodes[i].fillna(0)
#     fig, ax = plt.subplots(figsize=(12, 12))
#     gdf_zipcodes.plot(ax=ax, column=i, cmap='viridis', legend=True,
#                     legend_kwds={'label': f"Number of {i} by ZIP Code", 'orientation': "horizontal"},
#                     edgecolor='black', linewidth=0.8, alpha=0.8)  # Thicker black outlines for ZIP codes
#     gdf_zipcodes[gdf_zipcodes[i] == 0].plot(ax=ax, color='lightgray', edgecolor='gray', linewidth=0.2)
#     plt.title(f'Number of {i} of Each ZIP Code in California', fontsize=16)
#     plt.xlabel('Longitude', fontsize=12)
#     plt.ylabel('Latitude', fontsize=12)
#     plt.show()


Census Tract Plotting


In [ ]:
# census
print(storage_der["nameplate_capacity_mw"].describe())
print(storage_der.groupby('zip_code')['nameplate_capacity_mw'].sum().describe())

zip_counts = storage_der['zip_code'].value_counts()
top_10_zipcodes = zip_counts.head(20)
plt.figure(figsize=(10, 6))
top_10_zipcodes.plot(kind='bar')
plt.title('Top 20 Most Frequent ZIP Codes')
plt.xlabel('ZIP Code')
plt.ylabel('Frequency')
plt.xticks(rotation=45)
plt.show()


aggregated_storage_der = storage_der.groupby('zip_code')['nameplate_capacity_mw'].sum().sort_values(ascending=False).head(20)
aggregated_storage_der.plot(kind='bar')
plt.title('Top 20 Highest Capacity ZIP Codes')
plt.xlabel('ZIP Code')
plt.ylabel('Capacity (MW)')
plt.xticks(rotation=45)
plt.show()


aggregated_storage_der = storage_der.groupby('zip_code')['nameplate_capacity_mw'].sum().reset_index()
zip_shapefile = '/Users/danielleknutson/Downloads/tl_2024_us_zcta520/tl_2024_us_zcta520.shp'
gdf_zipcodes = gpd.read_file(zip_shapefile)
states = gpd.read_file("https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json")
california = states[states['name'] == 'California']
gdf_zipcodes = gdf_zipcodes[gdf_zipcodes.geometry.within(california.geometry.iloc[0])]
gdf_zipcodes['ZCTA5CE20'] = gdf_zipcodes['ZCTA5CE20'].astype(str).str.zfill(5)
aggregated_storage_der['zip_code'] = aggregated_storage_der['zip_code'].astype(str).str.zfill(5)
gdf_zipcodes = gdf_zipcodes.merge(aggregated_storage_der, left_on='ZCTA5CE20', right_on='zip_code', how='left')
gdf_zipcodes['nameplate_capacity_mw'] = gdf_zipcodes['nameplate_capacity_mw'].fillna(0)
fig, ax = plt.subplots(figsize=(12, 12))
gdf_zipcodes.plot(ax=ax, column='nameplate_capacity_mw', cmap='viridis', legend=True,
                  legend_kwds={'label': "Capacity by ZIP Code",
                               'orientation': "horizontal"},
                  edgecolor='gray', linewidth=0.5, alpha=0.8)
gdf_zipcodes[gdf_zipcodes['nameplate_capacity_mw'] == 0].plot(ax=ax, color='lightgray', edgecolor='gray', linewidth=0.2)
plt.title('California ZIP Code Capacity Distribution', fontsize=16)
plt.xlabel('Longitude', fontsize=12)
plt.ylabel('Latitude', fontsize=12)
plt.show()


zip_code_counts = storage_der['zip_code'].value_counts().reset_index()
zip_code_counts.columns = ['zip_code', 'occurrences']
zip_shapefile = '/Users/danielleknutson/Downloads/tl_2024_us_zcta520/tl_2024_us_zcta520.shp'
gdf_zipcodes = gpd.read_file(zip_shapefile)
gdf_zipcodes['ZCTA5CE20'] = gdf_zipcodes['ZCTA5CE20'].astype(str)  # Ensure matching formats
zip_code_counts['zip_code'] = zip_code_counts['zip_code'].astype(str)  # Ensure matching formats
gdf_zipcodes = gdf_zipcodes.merge(zip_code_counts, left_on='ZCTA5CE20', right_on='zip_code', how='left')
fig, ax = plt.subplots(figsize=(12, 12))
gdf_zipcodes.plot(ax=ax, column='occurrences', cmap='viridis', legend=True,
                  legend_kwds={'label': "Number of Occurrences by ZIP Code",
                               'orientation': "horizontal"},
                  edgecolor='gray', linewidth=0.5, alpha=0.8)
plt.title('Number of Occurrences of Each ZIP Code in California', fontsize=16)
plt.xlabel('Longitude', fontsize=12)
plt.ylabel('Latitude', fontsize=12)
plt.show()
print(len(storage_der["latitude"].unique()))

print(len(storage_der["census_tract"].unique()))

print(len(storage_der["zip_code"].unique()))



# # states = gpd.read_file("https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json")
# # california = states[states['name'] == 'California']
# # gdf_census_tract = gdf_census_tract[gdf_census_tract.geometry.within(california.geometry.iloc[0])]
# # gdf_census_tract['GEOID'] = gdf_census_tract['GEOID'].astype(str)
# # aggregated_storage_der['census_tract'] = aggregated_storage_der['census_tract'].astype(str)
# ###
# merged_data = acs_data.merge(aggregated_storage_der, left_on='tract', right_on='TRACTCE_only', how='left')

# merged_data['total_population'] = merged_data['total_population'].replace(0, 1200).fillna(1200)
# merged_data['der_capacity_per_capita'] = merged_data['nameplate_capacity_mw'] / merged_data['total_population']
# ###
# gdf_census_tract = gdf_census_tract.merge(merged_data, left_on='GEOID', right_on='census_tract')
# print(gdf_census_tract.head(2))

# gdf_census_tract['der_capacity_per_capita'] = gdf_census_tract['der_capacity_per_capita'].fillna(0)
# fig, ax = plt.subplots(figsize=(12, 12))
# gdf_census_tract.plot(ax=ax, column='der_capacity_per_capita', cmap='viridis', legend=True,
#                   legend_kwds={'label': "Per Capita DER Capacity by census_tract",
#                                'orientation': "horizontal"},
#                   edgecolor='gray', linewidth=0.5, alpha=0.8)
# plt.title('California census_tract Capacity Distribution', fontsize=16)
# plt.xlabel('Longitude', fontsize=12)
# plt.ylabel('Latitude', fontsize=12)
# plt.show()



# plotbigpercapita = merged_data.groupby('census_tract')['der_capacity_per_capita'].sum().sort_values(ascending=False).head(20)
# plotbigpercapita.plot(kind='bar')
# plt.title('Top 20 Highest Per Capita Capacity census_tract')
# plt.xlabel('census_tract')
# plt.ylabel('Capacity (MW)')
# plt.xticks(rotation=45)
# plt.show()


In [ ]:
# print(uswtdb_wind_der["project_capacity_mw"].describe())
# print(uswtdb_wind_der.groupby('census_tract')['project_capacity_mw'].sum().describe())

# zip_counts = uswtdb_wind_der['census_tract'].value_counts()
# top_10_zipcodes = zip_counts.head(20)
# plt.figure(figsize=(10, 6))
# top_10_zipcodes.plot(kind='bar')
# plt.title('Top 20 Most Frequent census_tract')
# plt.xlabel('census_tract')
# plt.ylabel('Frequency')
# plt.xticks(rotation=45)
# plt.show()


# aggregated_uswtdb_wind_der = uswtdb_wind_der.groupby('census_tract')['project_capacity_mw'].sum().sort_values(ascending=False).head(20)
# aggregated_uswtdb_wind_der.plot(kind='bar')
# plt.title('Top 20 Highest Capacity census_tract')
# plt.xlabel('census_tract')
# plt.ylabel('Capacity (MW)')
# plt.xticks(rotation=45)
# plt.show()


# aggregated_uswtdb_wind_der = uswtdb_wind_der.groupby('census_tract')['project_capacity_mw'].sum().reset_index()
# aggregated_uswtdb_wind_der['census_tract'] = aggregated_uswtdb_wind_der['census_tract'].astype(str)
# gdf_census_tract = gdf_census_tract.merge(aggregated_uswtdb_wind_der, left_on='GEOID', right_on='census_tract', how='left', suffixes=('', '_wind'))
# gdf_census_tract['project_capacity_mw'] = gdf_census_tract['project_capacity_mw'].fillna(0)
# print(len(gdf_census_tract[gdf_census_tract['project_capacity_mw'] != 0]))
# fig, ax = plt.subplots(figsize=(12, 12))
# gdf_census_tract.plot(ax=ax, column='project_capacity_mw', cmap='viridis', legend=True,
#                   legend_kwds={'label': "Capacity by ZIP Code",
#                                'orientation': "horizontal"},
#                   edgecolor='gray', linewidth=0.5, alpha=0.8)
# gdf_census_tract[gdf_census_tract['project_capacity_mw'] == 0].plot(ax=ax, color='lightgray', edgecolor='gray', linewidth=0.2)
# plt.title('California census_tract Capacity Distribution', fontsize=16)
# plt.xlabel('Longitude', fontsize=12)
# plt.ylabel('Latitude', fontsize=12)
# plt.show()

# census_tract_counts_wind = uswtdb_wind_der['census_tract'].value_counts().reset_index()
# census_tract_counts_wind.columns = ['census_tract', 'occurrences']
# census_tract_counts_wind = census_tract_counts_wind.rename(columns={'census_tract': 'GEOID'})
# gdf_census_tract['GEOID'] = gdf_census_tract['GEOID'].astype(str)  # Ensure the formats match
# census_tract_counts_wind['GEOID'] = census_tract_counts_wind['GEOID'].astype(str)  # Ensure the formats match
# gdf_census_tract = gdf_census_tract.merge(census_tract_counts_wind, on='GEOID', how='left', suffixes=('', '_wind'))
# gdf_census_tract['occurrences_wind'] = gdf_census_tract['occurrences_wind'].fillna(0)
# fig, ax = plt.subplots(figsize=(12, 12))
# gdf_census_tract.plot(ax=ax, column='occurrences_wind', cmap='viridis', legend=True,
#                   legend_kwds={'label': "Number of Occurrences by census_tract", 'orientation': "horizontal"},
#                   edgecolor='black', linewidth=0.8, alpha=0.8)  # Thicker black outlines for ZIP codes
# gdf_census_tract[gdf_census_tract['occurrences_wind'] == 0].plot(ax=ax, color='lightgray', edgecolor='gray', linewidth=0.2)
# plt.title('Number of Occurrences of Each census_tract in California', fontsize=16)
# plt.xlabel('Longitude', fontsize=12)
# plt.ylabel('Latitude', fontsize=12)
# plt.show()

In [ ]:
# aggregated_uswtdb_wind_der['TRACTCE_only'] = aggregated_uswtdb_wind_der['census_tract'].astype(str).str[-6:]

# merged_data = acs_data.merge(aggregated_uswtdb_wind_der, left_on='tract', right_on='TRACTCE_only', how='left')

# merged_data['total_population'] = merged_data['total_population'].replace(0, 1200).fillna(1200)
# merged_data['der_capacity_per_capita'] = merged_data['project_capacity_mw'] / merged_data['total_population']

# gdf_census_tract = gdf_census_tract.merge(merged_data, left_on='GEOID', right_on='census_tract', how='left', suffixes=('', '_wind'))
# gdf_census_tract['der_capacity_per_capita'] = gdf_census_tract['der_capacity_per_capita'].fillna(0)
# fig, ax = plt.subplots(figsize=(12, 12))
# gdf_census_tract.plot(ax=ax, column='der_capacity_per_capita', cmap='viridis', legend=True,
#                   legend_kwds={'label': "DER Capacity per Capita by ZIP Code",
#                                'orientation': "horizontal"},
#                   edgecolor='gray', linewidth=0.5, alpha=0.8)
# gdf_census_tract[gdf_census_tract['der_capacity_per_capita'] == 0].plot(ax=ax, color='lightgray', edgecolor='gray', linewidth=0.2)

# plt.title('California ZIP Code Wind Turbine DER Capacity per Capita Distribution', fontsize=16)
# plt.xlabel('Longitude', fontsize=12)
# plt.ylabel('Latitude', fontsize=12)
# plt.show()